In [1]:
!pip install torch torch-geometric scikit-learn networkx

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import random

from torch_geometric.datasets import PPI
from sklearn.metrics import roc_auc_score

In [3]:
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
train_dataset = PPI(root='/tmp/PPI', split='train')
test_dataset = PPI(root='/tmp/PPI', split='test')

train_data = train_dataset[0]
test_data = test_dataset[0]

print(train_data)
print(test_data)

Data(x=[1767, 50], edge_index=[2, 32318], y=[1767, 121])
Data(x=[3224, 50], edge_index=[2, 100648], y=[3224, 121])


In [5]:
def compute_entropy(G, node):
    dist = nx.single_source_shortest_path_length(G, node)
    vals = list(dist.values())

    counts = np.bincount(vals)
    probs = counts / counts.sum()

    entropy = -sum(
        p * np.log(p + 1e-9)
        for p in probs if p > 0
    )

    return entropy

In [17]:
def build_anchor_sets(data, fixed_num_sets=8):
    G = nx.Graph()

    edge_index = data.edge_index.cpu().numpy()

    for i in range(edge_index.shape[1]):
        u = int(edge_index[0, i])
        v = int(edge_index[1, i])
        G.add_edge(u, v)

    nodes = sorted(list(G.nodes()))
    num_nodes = len(nodes)

    entropy_scores = {}

    for node in nodes:
        entropy_scores[node] = compute_entropy(G, node)

    ranked = sorted(
        entropy_scores,
        key=entropy_scores.get,
        reverse=True
    )

    anchor_sets = []

    for i in range(fixed_num_sets):
        c = max(1, num_nodes // (2 ** (i + 1)))
        anchor_sets.append(ranked[:c])

    return anchor_sets, G, nodes

In [18]:
train_anchor_sets, train_G, train_nodes = build_anchor_sets(
    train_data,
    fixed_num_sets=8
)

print([len(s) for s in train_anchor_sets])

[779, 389, 194, 97, 48, 24, 12, 6]


In [19]:
def compute_anchor_features(data, anchor_sets, G, nodes):
    node_to_idx = {
        node: idx
        for idx, node in enumerate(nodes)
    }

    features = np.zeros((data.num_nodes, len(anchor_sets)))

    for node in nodes:
        idx = node_to_idx[node]

        for j, anchors in enumerate(anchor_sets):
            sims = []

            for a in anchors:
                try:
                    d = nx.shortest_path_length(G, node, a)
                    sims.append(1.0 / (d + 1))
                except:
                    sims.append(0.0)

            features[idx, j] = np.mean(sims)

    return torch.tensor(features, dtype=torch.float)

In [20]:
train_anchor_feat = compute_anchor_features(
    train_data,
    train_anchor_sets,
    train_G,
    train_nodes
)

print(train_anchor_feat.shape)

torch.Size([1767, 8])


In [21]:
class IEA_GNN_F_2L(nn.Module):
    def __init__(self, in_dim, anchor_dim, hidden=64):
        super().__init__()

        self.lin1 = nn.Linear(
            in_dim + anchor_dim,
            hidden
        )

        self.lin2 = nn.Linear(
            hidden,
            hidden
        )

        self.dropout = nn.Dropout(0.5)

    def aggregate(self, x, edge_index):
        row, col = edge_index

        out = torch.zeros_like(x)
        out.index_add_(0, row, x[col])

        deg = torch.bincount(
            row,
            minlength=x.size(0)
        ).float().unsqueeze(1)

        deg[deg == 0] = 1

        return out / deg

    def forward(self, x, edge_index):
        h = self.aggregate(x, edge_index)
        h = F.relu(self.lin1(h))
        h = self.dropout(h)

        h = self.aggregate(h, edge_index)
        h = self.lin2(h)

        return h

In [22]:
model = IEA_GNN_F_2L(
    train_data.x.shape[1],
    train_anchor_feat.shape[1],
    hidden=64
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [23]:
train_data = train_data.to(device)
test_data = test_data.to(device)
train_anchor_feat = train_anchor_feat.to(device)

In [24]:
def edge_samples(data, num_samples=50000):
    edge_index = data.edge_index

    device = edge_index.device

    pos_idx = torch.randint(
        0,
        edge_index.shape[1],
        (num_samples,),
        device=device
    )

    pos_edges = edge_index[:, pos_idx]

    neg_edges = torch.randint(
        0,
        data.num_nodes,
        (2, num_samples),
        device=device
    )

    edges = torch.cat(
        [pos_edges, neg_edges],
        dim=1
    )

    labels = torch.cat([
        torch.ones(num_samples, device=device),
        torch.zeros(num_samples, device=device)
    ])

    return edges, labels

In [25]:
for epoch in range(1000):
    model.train()
    optimizer.zero_grad()

    x = torch.cat(
        [train_data.x, train_anchor_feat],
        dim=1
    )

    z = model(x, train_data.edge_index)

    edges, labels = edge_samples(train_data)

    pred = (
        z[edges[0]] *
        z[edges[1]]
    ).sum(dim=1)

    loss = F.binary_cross_entropy_with_logits(
        pred,
        labels
    )

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(epoch, loss.item())

0 0.7094436287879944
100 0.6023380756378174
200 0.5917540192604065
300 0.5880481004714966
400 0.5833984017372131
500 0.5932419896125793
600 0.5853663086891174
700 0.5793142914772034
800 0.5759609341621399
900 0.5756321549415588


In [26]:
test_anchor_sets, test_G, test_nodes = build_anchor_sets(
    test_data,
    fixed_num_sets=8
)

test_anchor_feat = compute_anchor_features(
    test_data,
    test_anchor_sets,
    test_G,
    test_nodes
).to(device)

In [27]:
model.eval()

x_test = torch.cat(
    [test_data.x, test_anchor_feat],
    dim=1
)

with torch.no_grad():
    z = model(x_test, test_data.edge_index)

    edges, labels = edge_samples(test_data)

    pred = torch.sigmoid(
        (z[edges[0]] * z[edges[1]]).sum(dim=1)
    ).cpu().numpy()

labels = labels.cpu().numpy()

auc = roc_auc_score(labels, pred)

print("IEA-GNN-F-2L PPI AUC:", auc)

IEA-GNN-F-2L PPI AUC: 0.710076056
